# Baseline VLM inference skeleton (Colab / Kaggle, T4)

Implements the resumable-inference pattern from `configs/compute_plan.md`:
incremental JSONL writes, skip-if-done, 4-bit NF4 loading for 7B+ models.
Runs against `tests/fixtures/` by default so it's testable without any real
dataset downloaded yet — point `IMAGE_DIR`/`DOC_IDS` at real data once available.

**Fill in `MODEL_ID` and `run_inference_on_image()` for your chosen model**
(see the candidate table in `configs/compute_plan.md`) — everything else
(checkpointing, resume logic, schema validation) is meant to be reused as-is.

In [ ]:
# One-time setup (Colab). On Kaggle, prefer attaching the repo as a Dataset/Notebook input instead of git-cloning.
# !git clone <your-repo-url> repo && cd repo
!pip install -q -e ".[vlm]"

from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/invoice-extract-checkpoints'  # survives disconnects; adjust for Kaggle to /kaggle/working

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, 'src')

MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'  # start here per configs/compute_plan.md; swap in the 7B id + four_bit=True if needed
USE_4BIT = False  # flip to True for 7B+ models

PRED_OUT_PATH = Path(CHECKPOINT_DIR) / 'predictions.jsonl'
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
from invoice_extract.utils.quantization import load_quantized_vlm

model, processor = load_quantized_vlm(MODEL_ID, four_bit=USE_4BIT)

In [ ]:
EXTRACTION_PROMPT = '''Extract this invoice's fields as JSON matching exactly this shape \
(use null for anything illegible or absent, do not invent values):\n\n{schema_shape}'''

SCHEMA_SHAPE_HINT = json.dumps({
    'parties': {'vendor': {'name': None, 'address': None, 'gstin': None}, 'buyer': {'name': None}},
    'invoice_meta': {'invoice_number': None, 'invoice_date': 'YYYY-MM-DD', 'due_date': 'YYYY-MM-DD or null'},
    'line_items': [{'description': None, 'quantity': None, 'unit_price': None, 'line_total': None}],
    'totals': {'subtotal': None, 'tax_total': None, 'grand_total': None},
}, indent=2)


def run_inference_on_image(image_path: str) -> dict:
    """Fill in: build the chat-template messages for MODEL_ID, run generate(),
    parse the JSON substring out of the response, and return a dict with (at
    minimum) the leaf keys shown in SCHEMA_SHAPE_HINT above. Wrap the JSON
    parse in try/except -> return an all-null dict on failure, never raise,
    so one bad generation doesn't kill a long batch run."""
    raise NotImplementedError('Implement per MODEL_ID chosen above; see configs/compute_plan.md candidate table')

In [ ]:
def to_canonical(doc_id: str, source_dataset: str, split: str, raw_extraction: dict) -> dict:
    """Wrap a model's raw extraction into the canonical schema shape
    (source/doc_id/pages are inference-run metadata, not model output)."""
    return {
        'schema_version': '1.0.0',
        'doc_id': doc_id,
        'source': {'dataset': source_dataset, 'original_id': doc_id, 'split': split},
        'document_type': 'invoice',
        'pages': [{'page_index': 0, 'image_path': str(IMAGE_DIR / f'{doc_id}.png'), 'width': 0, 'height': 0}],
        'parties': raw_extraction.get('parties', {'vendor': {}}),
        'invoice_meta': raw_extraction.get('invoice_meta', {'invoice_number': None, 'invoice_date': None}),
        'line_items': raw_extraction.get('line_items', []),
        'tax_lines': raw_extraction.get('tax_lines', []),
        'totals': raw_extraction.get('totals', {'grand_total': None}),
    }

In [ ]:
# Resumable inference loop: skip doc_ids already written, flush after every doc.
IMAGE_DIR = Path('data/processed/gst_in_synthetic/images')  # point at real data once downloaded/generated
SOURCE_DATASET = 'gst_in_synthetic'
SPLIT = 'test'
DOC_IDS = [p.stem for p in sorted(IMAGE_DIR.glob('*.png'))] if IMAGE_DIR.exists() else []

already_done = set()
if PRED_OUT_PATH.exists():
    already_done = {json.loads(line)['doc_id'] for line in PRED_OUT_PATH.read_text().splitlines() if line.strip()}

with PRED_OUT_PATH.open('a') as f:
    for doc_id in DOC_IDS:
        if doc_id in already_done:
            continue
        raw = run_inference_on_image(str(IMAGE_DIR / f'{doc_id}.png'))
        canonical = to_canonical(doc_id, SOURCE_DATASET, SPLIT, raw)
        f.write(json.dumps(canonical) + '\n')
        f.flush()

print(f'Predictions written to {PRED_OUT_PATH}')

In [ ]:
# Score against the harness once predictions exist (swap --gold to a real gold file when available).
!python -m invoice_extract.eval.harness --gold tests/fixtures/gold.jsonl --pred tests/fixtures/pred.jsonl --by-source